# Prompt A/B Testing — Compare Prompt Variants Systematically
## Version Your Prompts — Systematic A/B Comparison
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/78-prompt-ab-testing/prompt_ab_testing_workbook.ipynb)

Runs the same 5 questions through two prompt variants (concise vs thorough) and scores responses by word count / 100, capped at 1.0. Identifies which variant wins per question and overall.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why prompt wording changes output dramatically; scoring strategies |
| 2 | **Variants** — VARIANT_A = concise one-sentence; VARIANT_B = thorough explanation |
| 3 | **score_response()** — word_count / 100, max 1.0 — proxy for depth/completeness |
| 4 | **Run Both Variants** — Same question through A then B in one LangGraph pass |
| 5 | **Winner Analysis** — Per-question winner + overall stats |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

VARIANT_A_PROMPT = "You are a concise assistant. Answer in exactly one short sentence: {question}"
VARIANT_B_PROMPT = "You are a thorough assistant. Provide a detailed, informative answer, explaining key concepts clearly: {question}"

TEST_INPUTS = [
    "What is photosynthesis?",
    "How does the internet work?",
    "What is quantum entanglement?",
    "Why is the sky blue?",
    "What is the difference between RAM and storage?",
]

def score_response(response: str) -> float:
    return round(min(len(response.split()) / 100, 1.0), 4)

class ABState(TypedDict):
    question: str; response_a: str; response_b: str; score_a: float; score_b: float

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def run_variant_a(state):
    text = llm.invoke([HumanMessage(content=VARIANT_A_PROMPT.format(question=state["question"]))]).content.strip()
    return {**state, "response_a": text, "score_a": score_response(text)}

def run_variant_b(state):
    text = llm.invoke([HumanMessage(content=VARIANT_B_PROMPT.format(question=state["question"]))]).content.strip()
    return {**state, "response_b": text, "score_b": score_response(text)}

g = StateGraph(ABState)
g.add_node("run_variant_a", run_variant_a)
g.add_node("run_variant_b", run_variant_b)
g.add_edge(START, "run_variant_a"); g.add_edge("run_variant_a", "run_variant_b"); g.add_edge("run_variant_b", END)
app = g.compile()

results = []
for q in TEST_INPUTS:
    r = app.invoke({"question": q, "response_a": "", "response_b": "", "score_a": 0.0, "score_b": 0.0})
    results.append(r)
    winner = "A" if r["score_a"] >= r["score_b"] else "B"
    print(f"Q: {q}")
    print(f"  A ({r['score_a']:.2f}): {r['response_a'][:80]}")
    print(f"  B ({r['score_b']:.2f}): {r['response_b'][:80]}")
    print(f"  Winner: Variant {winner}")
    print()

a_wins = sum(1 for r in results if r["score_a"] >= r["score_b"])
b_wins = len(results) - a_wins
avg_a  = sum(r["score_a"] for r in results) / len(results)
avg_b  = sum(r["score_b"] for r in results) / len(results)
print(f"Overall — A wins: {a_wins}/{len(results)}  B wins: {b_wins}/{len(results)}")
print(f"Avg score — A: {avg_a:.3f}  B: {avg_b:.3f}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*